In [108]:
import pandas as pd
import numpy as np
import json

In [79]:
manufactured = "Ut�ngy�rtott"
manufactured_correct = "Utángyártott"

In [86]:
# Initialize an empty list to hold the JSON objects
data = []

# Read the JSONL file line by line
with open("test.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        try:
            # Parse the JSON object and append to the list
            data.append(json.loads(line))
        except ValueError as e:
            print(f"Error decoding line: {line}\nError: {e}")

# Convert the list of JSON objects to a DataFrame
df = pd.DataFrame(data)

df.drop("url", axis=1, inplace=True)
df.drop("availability", axis=1, inplace=True)

# Add a new attribute named Originality
df['originality'] = df['product_name'].apply(lambda x: 'manufactured' if manufactured in x else 'original')

# Remove "manufactured" from product_name if it exists and handle extra spaces
df['product_name'] = df['product_name'].apply(
    lambda x: x.replace(manufactured, '').replace('  ', ' ').strip() if manufactured in x else x
)


# Display the first few rows of the dataframe
print(df)

     price          competitor product_name originality
0     5691      onlinetoner.hu   HP CZ101AE    original
1     5700           pcland.hu   HP CZ101AE    original
2     5840        firstshop.hu   HP CZ101AE    original
3     5691      onlinetoner.hu   HP CZ101AE    original
4     5700           pcland.hu   HP CZ101AE    original
...    ...                 ...          ...         ...
1094  6404  onlinepapirbolt.hu   HP N9K06AE    original
1095  6446       best-toner.hu   HP N9K06AE    original
1096  6450        bestmarkt.hu   HP N9K06AE    original
1097  6485   irodaitermekek.hu   HP N9K06AE    original
1098  6490             ipon.hu   HP N9K06AE    original

[1099 rows x 4 columns]


In [87]:
customer_df = pd.read_json("customer_request_test.json")

customer_df

,name,originality,customer_name,min_price,lower_min
0,Samsung MLT-D116L,manufactured,,200,10
1,Samsung MLT-D116L,original,,200,10
2,HP CZ101AE,original,,500,100
3,Canon CL-541XL Color (BS5226B005AA),original,,13250,100


In [88]:
# Check for mismatched values between product_name and name columns
mismatched_products = set(df["product_name"]) - set(customer_df["name"])

mismatched_products

{'Brother TN-1030 Black',
 'Brother TN-2421 Black',
 'Brother TZe-231',
 'Canon CL-546XL Color (BS8288B001AA)',
 'Canon KP-108IN Multipack (AJ3115B001AA)',
 'Canon PG-560 + CL-561 Multipack (3713C006AA)',
 'Canon PG-560-XL (3712C001AA)',
 'Canon PGI-570XL + CLI-571XL (0372C004)',
 'Canon PGI-580 + CLI-581 Multipack PGBK/C/M/Y/BK (2078C005AA)',
 'Canon PGI-580XXL / CLI-581XXL Multipack',
 'HP CF244A Black',
 'HP CF283A Black',
 'HP CH561EE Black',
 'HP CZ102AE',
 'HP F6U66AE',
 'HP N9K06AE',
 'HP W1106A',
 'HP W1106A (106A)',
 'HP W1350A',
 'HP W1350X',
 'Xerox 106R02773',
 'Xerox 106R04348'}

In [92]:
# Create masks to check if names and originality in one DataFrame are in the other
def check_common(row, other_df):
    matches = other_df[(other_df['name'] == row['product_name']) & (other_df['originality'] == row['originality'])]
    return len(matches) > 0

In [93]:
mask_df = df.apply(lambda row: check_common(row, customer_df), axis=1)

In [101]:
#mask_customer_df = customer_df.apply(lambda row: check_common(row, df), axis=1)

# Filter the DataFrames using the masks
common_in_df = df[mask_df]

#print(set(common_in_df["product_name"].tolist()))
common_in_df


,price,competitor,product_name,originality
0,5691,onlinetoner.hu,HP CZ101AE,original
1,5700,pcland.hu,HP CZ101AE,original
2,5840,firstshop.hu,HP CZ101AE,original
3,5691,onlinetoner.hu,HP CZ101AE,original
4,5700,pcland.hu,HP CZ101AE,original
...,...,...,...,...
994,3690,papir-bolt.hu,Samsung MLT-D116L,manufactured
995,3790,pixelrodeo.hu,Samsung MLT-D116L,manufactured
996,3790,tonerbolt.eu,Samsung MLT-D116L,manufactured
997,3990,emag.hu,Samsung MLT-D116L,manufactured


In [102]:
min_values = common_in_df.groupby(['product_name', 'originality']).min()

min_values

,,price,competitor
product_name,originality,,
Canon CL-541XL Color (BS5226B005AA),original,10224,alza.hu
HP CZ101AE,original,5480,alza.hu
Samsung MLT-D116L,manufactured,1900,aqua.hu


In [109]:
# Rename 'name' column in customer_df to 'product_name'
customer_df = customer_df.rename(columns={'name': 'product_name'})

# Merge the dataframes
merged_df = pd.merge(min_values, customer_df[['product_name', 'originality', 'min_price', 'lower_min']], on=['product_name', 'originality'])

# Convert 'price' and 'lower_min' to numeric types
merged_df['price'] = pd.to_numeric(merged_df['price'], errors='coerce')
merged_df['lower_min'] = pd.to_numeric(merged_df['lower_min'], errors='coerce')

# Lower the price by 'lower_min' amount
merged_df['lowered_price'] = merged_df['price'] - merged_df['lower_min']

# If 'lowered_price' is less than 'min_price', set 'lowered_price' to 'min_price'
merged_df['lowered_price'] = np.where(merged_df['lowered_price'] < merged_df['min_price'], merged_df['min_price'], merged_df['lowered_price'])

merged_df

,product_name,originality,price,competitor,min_price,lower_min,lowered_price
0,Canon CL-541XL Color (BS5226B005AA),original,10224,alza.hu,13250,100,13250
1,HP CZ101AE,original,5480,alza.hu,500,100,5380
2,Samsung MLT-D116L,manufactured,1900,aqua.hu,200,10,1890


In [110]:

merged_df.to_xml('customer_min_prices.xml')